# Compile cNMF Evaluation Results into Excel — IGVF Hon WTC11

This notebook collects all evaluation outputs (enrichment, perturbation association, explained variance, etc.)
for the `030726_20iter_5KHVG_torch_halsvar_batch_e7` cNMF run and compiles them into a single multi-sheet Excel workbook.

**Key differences from the IGVF ccperturbseq reference notebook:**
- Evaluation results are in `Eval/` (not `Evaluation/`)
- Categorical key is `batch` (4 IGVF batches), not `timepoint`
- Guide data is already embedded in h5mu (no separate guide file)
- Perturbation analysis uses WTC (combined) sample only

In [1]:
import pandas as pd
import muon as mu
import sys
import os

sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')

from Stage3_Interpretation.B_Summarization.src import (
    compile_Program_loading_score_sheet_long, compile_Program_loading_score_sheet_flat,
    Compile_GO_sheet, Compile_Geneset_sheet, Compile_Trait_sheet,
    Compile_Perturbation_sheet, Compile_Association_sheet, Compile_Explained_variance,
    Compile_Target_Summary_sheet, Compile_Summary_sheet,
    add_specificity_scores_file, check_program_name_match
)

/home/users/ymo/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1. Set Parameters

In [2]:
# ── cNMF output paths ──
project_root = '/oak/stanford/groups/engreitz/Users/ymo/Project/IGVF_Hon_WTC11'
out_dir = f'{project_root}/Result'
run_name = '030726_20iter_5KHVG_torch_halsvar_batch_e7'

# ── Evaluation results directory (this project uses 'Eval/', not 'Evaluation/') ──
eval_dir_name = 'Eval'

# ── MuData keys ──
prog_key = 'cNMF'
data_key = 'rna'
gene_names_key = 'symbol'
categorical_key = 'WTC'
guide_targets_key = 'guide_targets'

# ── Batch values (from obs['batch']) ──
batch_values = ['WTC']

# ── Compilation settings ──
num_gene = 300
components = [50]
sel_threshs = [2.0]
perturbation_file_name = 'perturbation_association_results'
non_targeting_key = ['non-targeting']
effect_size = 'log2FC'

# ── Column name keys ──
GO_Term_key = 'Term'
GO_Genes_key = 'Genes'
Geneset_Term_key = 'Term'
Geneset_Genes_key = 'Genes'
Trait_Term_key = 'Term'
Trait_Genes_key = 'Genes'
Perturbation_Sample_key = 'Sample'
adjusted_pval_key = 'Adjusted P-value'

## Step 2. Compile Excel

For each (K, threshold):
1. Load MuData (guide data already embedded)
2. Load evaluation result DataFrames from `Eval/` directory
3. Validate program name consistency
4. Build Target Summary and Summary sheets
5. Write all sheets to Excel

In [3]:
for sel_thresh in sel_threshs:
    for k in components:

        thresh_str = str(sel_thresh).replace('.', '_')
        eval_base = f'{out_dir}/{run_name}/{eval_dir_name}/{k}_{thresh_str}'
        output_folder = f'{out_dir}/{run_name}/Interpretation/Summary_table/{k}_{thresh_str}'
        os.makedirs(output_folder, exist_ok=True)

        # ── 2a. Load MuData (guide data already embedded) ──
        mdata_path = f'{out_dir}/{run_name}/adata/cNMF_{k}_{thresh_str}.h5mu'
        print(f'Loading MuData: {mdata_path}')
        mdata = mu.read(mdata_path)

        # ── 2b. Load evaluation DataFrames from Eval/ ──
        GO_path = f'{eval_base}/{k}_GO_term_enrichment.txt'
        Geneset_path = f'{eval_base}/{k}_geneset_enrichment.txt'
        Trait_path = f'{eval_base}/{k}_trait_enrichment.txt'
        Perturbation_path_base = f'{eval_base}/{k}_{perturbation_file_name}'
        Association_path = f'{eval_base}/{k}_categorical_association_results.txt'
        Explained_Variance_path = f'{eval_base}/{k}_Explained_Variance.txt'

        # Program loadings
        df_Program_loading_long = compile_Program_loading_score_sheet_long(mdata, num_gene=num_gene, data_key=data_key, gene_names_key=gene_names_key)
        df_Program_loading_flat = compile_Program_loading_score_sheet_flat(mdata, num_gene=num_gene, data_key=data_key, gene_names_key=gene_names_key)

        # Enrichment sheets
        df_GO = Compile_GO_sheet(GO_path, term_key=GO_Term_key, genes_key=GO_Genes_key) if os.path.exists(GO_path) else None
        df_Geneset = Compile_Geneset_sheet(Geneset_path, term_key=Geneset_Term_key, genes_key=Geneset_Genes_key) if os.path.exists(Geneset_path) else None
        df_Trait = Compile_Trait_sheet(Trait_path, term_key=Trait_Term_key, genes_key=Trait_Genes_key) if os.path.exists(Trait_path) else None

        # Perturbation (WTC only)
        perturbation_files = [f'{Perturbation_path_base}_{samp}.txt' for samp in batch_values]
        if any(os.path.exists(f) for f in perturbation_files):
            df_Perturbation = Compile_Perturbation_sheet(Perturbation_path_base, Sample=batch_values, sample_key=Perturbation_Sample_key)
            df_Perturbation_significant_gene_only = df_Perturbation[df_Perturbation['adj_pval'] < 0.05]
        else:
            print(f'No perturbation files found for: {Perturbation_path_base}')
            df_Perturbation = None
            df_Perturbation_significant_gene_only = None

        # Categorical association & explained variance
        df_Association = Compile_Association_sheet(Association_path) if os.path.exists(Association_path) else None
        df_Explained_Variance = Compile_Explained_variance(Explained_Variance_path) if os.path.exists(Explained_Variance_path) else None

        # ── 2c. Validate program names ──
        check_program_name_match(mdata, prog_key=prog_key, dataframes=[
            df_GO, df_Geneset, df_Trait, df_Perturbation,
            df_Association, df_Explained_Variance, df_Perturbation_significant_gene_only
        ])

        # ── 2d. Build Target Summary ──
        # Uses WTC perturbation + batch-level cell counts/expression
        df_Target_Summary = Compile_Target_Summary_sheet(
            mdata, Perturbation_path_base,
            Sample=batch_values, categorical_key=categorical_key,
            prog_key=prog_key, data_key=data_key,
            guide_targets_key=guide_targets_key,
            save_path=output_folder, effect_size=effect_size,
            gene_names_key=gene_names_key
        )

        # ── 2e. Build Summary sheet ──
        # Pass batch_values for 'Automatic Timepoint' (mean scores are per-batch);
        # perturbation columns use df_Perturbation['Sample'].unique() = ['WTC'] internally
        df_Summary = Compile_Summary_sheet(
            mdata, df_GO, df_Geneset, df_Perturbation, df_Program_loading_flat, df_Explained_Variance,
            Sample=batch_values, specicicity_path=output_folder,
            categorical_key=categorical_key, non_tagerting_key=non_targeting_key,
            effect_size=effect_size, adjusted_pval_key=adjusted_pval_key
        )

        # ── 2e2. Save key DataFrames as separate TSV files ──
        df_Summary.to_csv(f'{output_folder}/Summary_{k}_{thresh_str}.tsv', sep='\t')
        df_Program_loading_long.to_csv(f'{output_folder}/Program_Loadings_{k}_{thresh_str}.tsv', sep='\t')
        df_Target_Summary.to_csv(f'{output_folder}/Targets_Summary_{k}_{thresh_str}.tsv', sep='\t')
        print(f'Saved separate TSV files to {output_folder}')

        # ── 2f. Write Excel ──
        print(f'Compiling Excel for K={k}, threshold={sel_thresh}')
        MAX_ROWS = 1048575
        excel_path = f'{output_folder}/cNMF_{k}_{thresh_str}.xlsx'

        with pd.ExcelWriter(excel_path) as writer:
            df_Summary.to_excel(writer, sheet_name='Summary', index=True)
            df_Program_loading_long.to_excel(writer, sheet_name='Program Loadings', index=True)
            df_Target_Summary.to_excel(writer, sheet_name='Targets Summary', index=True)

            if df_Association is not None:
                df_Association.to_excel(writer, sheet_name='Sample Association', index=True)

            if df_Perturbation is not None:
                # Re-load with specificity scores
                combined_conditions = []
                for samp in batch_values:
                    df_Perturbation_ = add_specificity_scores_file(output_folder, Perturbation_path_base, samp)
                    df_Perturbation_[Perturbation_Sample_key] = samp
                    combined_conditions.append(df_Perturbation_)
                df_Perturbation_with_spec = pd.concat(combined_conditions)

                for i in range(0, len(df_Perturbation_with_spec), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation_with_spec.iloc[i:i+MAX_ROWS].to_excel(
                        writer, sheet_name=f'Perturbation Association {sheet_num}', index=True)

                for i in range(0, len(df_Perturbation_significant_gene_only), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation_significant_gene_only.iloc[i:i+MAX_ROWS].to_excel(
                        writer, sheet_name=f'significant regulators only {sheet_num}', index=True)

            if df_Trait is not None:
                df_Trait.to_excel(writer, sheet_name='Trait Enrichment', index=True)
            if df_GO is not None:
                df_GO.to_excel(writer, sheet_name='GO Term Enrichment', index=True)
            if df_Geneset is not None:
                df_Geneset.to_excel(writer, sheet_name='Geneset Enrichment', index=True)

        print(f'Done. Saved to {excel_path}')

Loading MuData: /oak/stanford/groups/engreitz/Users/ymo/Project/IGVF_Hon_WTC11/Result/030726_20iter_5KHVG_torch_halsvar_batch_e7/adata/cNMF_50_2_0.h5mu


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Load program loadings data in long form


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Load program loadings data in flat form
Load GO data
Load geneset data
Load trait data
Load perturbation data
Load categorical associa data
Load explained variance data
Generate target summary sheet
Get targeted gene mean expressuion per day
Compute guides cells per days
Compile significant programs
Compute significant programs
Calculate specificity scores for program
Compute correlation for WTC
Compute KD efficiency per target per condition
complie summary sheet


/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Stage3_Interpretation/B_Summarization/src/Compile_excel_sheet.py:893: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_mean = df_cell_program.groupby("cell_type")["expression"].mean()
/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Stage3_Interpretation/B_Summarization/src/Compile_excel_sheet.py:894: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_frac = df_cell_program.groupby("cell_type")["expression"].apply(
/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Stage3_Interpretation/B_Summarization/src/Compile_excel_sheet.py:

Saved separate TSV files to /oak/stanford/groups/engreitz/Users/ymo/Project/IGVF_Hon_WTC11/Result/030726_20iter_5KHVG_torch_halsvar_batch_e7/Interpretation/Summary_table/50_2_0
Compiling Excel for K=50, threshold=2.0
Done. Saved to /oak/stanford/groups/engreitz/Users/ymo/Project/IGVF_Hon_WTC11/Result/030726_20iter_5KHVG_torch_halsvar_batch_e7/Interpretation/Summary_table/50_2_0/cNMF_50_2_0.xlsx
